# 01 - Industry Data Generation for Marketing Mix Modeling

## Objective

Build the weekly MMM panel from packaged raw source files in `data/mmm_raw_data.zip`
(marketing spend, sales, pricing, holidays, weather, competitor) instead of simulating
media spend with random numbers.

### Output

- `data/raw/marketing_mix_data.csv` — one row per week for the rest of the pipeline

In [ ]:
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "mmm_raw_data.zip").exists() and (ROOT.parent / "data" / "mmm_raw_data.zip").exists():
    ROOT = ROOT.parent

RAW = ROOT / "data" / "raw"
SOURCE_DIR = RAW / "sources"
ZIP_PATH = ROOT / "data" / "mmm_raw_data.zip"

RAW.mkdir(parents=True, exist_ok=True)
SOURCE_DIR.mkdir(parents=True, exist_ok=True)

if not ZIP_PATH.exists():
    raise FileNotFoundError(f"Raw data ZIP not found: {ZIP_PATH}")

with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    zf.extractall(SOURCE_DIR)

print("Project root:", ROOT)
print("Extracted sources to:", SOURCE_DIR)
print("Files:", sorted(p.name for p in SOURCE_DIR.iterdir()))

## Load Media Spend

Weekly channel spend comes from `marketing_data.xlsx` (not `np.random.randint`).

In [ ]:
MEDIA_CHANNELS = [
    "Google_Search", "Google_Display", "Meta", "Instagram",
    "YouTube", "TV", "Radio", "Influencer", "Affiliate", "Email",
]

marketing = pd.read_excel(SOURCE_DIR / "marketing_data.xlsx", sheet_name="marketing_data")
marketing["Week"] = pd.to_datetime(marketing["Week"])

missing_media = [c for c in MEDIA_CHANNELS if c not in marketing.columns]
if missing_media:
    raise ValueError(f"Missing media columns in marketing_data.xlsx: {missing_media}")

media = marketing[["Week"] + MEDIA_CHANNELS].copy()
print(media.shape)
media.head()

## Load Business Drivers

Join sales, pricing, weather, holidays, and competitor spend on `Week`.

In [ ]:
sales = pd.read_csv(SOURCE_DIR / "sales.csv", parse_dates=["Week"])
pricing = pd.read_csv(SOURCE_DIR / "pricing.csv", parse_dates=["Week"])
competitor = pd.read_csv(SOURCE_DIR / "competitor.csv", parse_dates=["Week"])
holidays = pd.read_csv(SOURCE_DIR / "holidays.csv", parse_dates=["Date"])
weather = pd.read_csv(SOURCE_DIR / "weather.csv", parse_dates=["Date"])

pricing_weekly = (
    pricing.groupby("Week", as_index=False)
    .agg(Price=("Selling_Price", "mean"), Discount=("Discount", "mean"))
)

competitor_weekly = (
    competitor.groupby("Week", as_index=False)
    .agg(Competitor_Spend=("Competitor_Spend", "sum"))
)

holidays_weekly = holidays.rename(columns={"Date": "Week", "Holiday_Flag": "Holiday"})[
    ["Week", "Holiday"]
].copy()

weather_weekly = weather.rename(columns={"Date": "Week"})[["Week", "Temperature"]].copy()

sales_weekly = sales[["Week", "Sales", "Revenue", "Orders"]].copy()

print("sales:", sales_weekly.shape)
print("pricing:", pricing_weekly.shape)
print("competitor:", competitor_weekly.shape)
print("holidays:", holidays_weekly.shape)
print("weather:", weather_weekly.shape)

## Assemble Final Weekly Dataset

In [ ]:
df = media.copy()
df = df.merge(sales_weekly, on="Week", how="inner")
df = df.merge(pricing_weekly, on="Week", how="left")
df = df.merge(weather_weekly, on="Week", how="left")
df = df.merge(competitor_weekly, on="Week", how="left")
df = df.merge(holidays_weekly, on="Week", how="left")

df = df.sort_values("Week").reset_index(drop=True)

# Fill occasional join gaps with sensible defaults
df["Discount"] = df["Discount"].fillna(0)
df["Holiday"] = df["Holiday"].fillna(0).astype(int)
df["Price"] = df["Price"].fillna(df["Price"].median())
df["Temperature"] = df["Temperature"].fillna(df["Temperature"].median())
df["Competitor_Spend"] = df["Competitor_Spend"].fillna(df["Competitor_Spend"].median())

final_cols = [
    "Week",
    *MEDIA_CHANNELS,
    "Discount",
    "Price",
    "Temperature",
    "Competitor_Spend",
    "Holiday",
    "Sales",
    "Revenue",
    "Orders",
]
df = df[final_cols]

print(df.shape)
print("Date range:", df["Week"].min().date(), "→", df["Week"].max().date())
df.head()

## Visualize Sales

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(df["Week"], df["Sales"])
plt.title("Weekly Sales (from sales.csv)")
plt.xlabel("Week")
plt.ylabel("Sales")
plt.grid(True)
plt.show()

## Save Dataset

In [ ]:
output = RAW / "marketing_mix_data.csv"
df.to_csv(output, index=False)

print("Saved to:", output)
print("Rows:", len(df), "| Columns:", len(df.columns))

# Business Interpretation

This notebook now loads real packaged sources:

- **Media spend** ← `marketing_data.xlsx`
- **Sales / revenue / orders** ← `sales.csv`
- **Price / discount** ← `pricing.csv`
- **Temperature** ← `weather.csv`
- **Holiday flag** ← `holidays.csv`
- **Competitor spend** ← `competitor.csv`

No media columns are generated with `np.random.randint` anymore.
